# Drone flight paths along street networks

Author: [Vishwa](https://github.com/vishwagw)

  - [Documentation](https://osmnx.readthedocs.io/)
  - [Journal article and citation info](https://doi.org/10.1111/gean.70009)
  - [Code repository](https://github.com/gboeing/osmnx)
  - [Examples gallery](https://github.com/gboeing/osmnx-examples)

Small drones flying through cities are often planned to follow streets, staying over public right-of-way instead of cutting across buildings and private property. This notebook combines three things OSMnx can give you (a street network, node elevations from a raster, and building footprints with heights) to turn an ordinary street route into a 3D flight path, with a safe altitude at every waypoint.

This is a planning illustration, not certified flight-planning software. OpenStreetMap building heights are incomplete, and real flights must follow local aviation rules.

In [ ]:
#!uv pip install --system --quiet osmnx[all]
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
import pandas as pd
import rasterio
from rasterio.merge import merge
from shapely.geometry import LineString, Point

ox.settings.log_console = True
ox.__version__

## 1. Street network with node elevations

Download the drivable street network around an address, then attach node elevations from the local raster files included in this repo (see the [elevations example](12-node-elevations-edge-grades.ipynb) for details). We keep an unprojected copy for lat/lon lookups and a projected copy so that distances are in meters.

In [ ]:
address = "600 Montgomery St, San Francisco, California, USA"
dist = 500
G = ox.graph.graph_from_address(address, dist=dist, dist_type="bbox", network_type="drive")

raster_paths = ["./input_data/elevation1.tif", "./input_data/elevation2.tif"]
G = ox.elevation.add_node_elevations_raster(G, raster_paths, cpus=1)
G = ox.elevation.add_edge_grades(G, add_absolute=True)
Gp = ox.projection.project_graph(G)
crs = Gp.graph["crs"]
len(G), len(G.edges)

## 2. Building footprints and heights

Building footprints come from OpenStreetMap's `building` tag. A drone needs to know how tall each one is, so we estimate a height in meters from the `height` tag when it exists. Otherwise we use `building:levels` times an assumed storey height, and finally a conservative default. It's worth checking how many buildings fall into each case: the more defaults, the less you should trust the result.

In [ ]:
buildings = ox.features.features_from_address(address, tags={"building": True}, dist=dist)
buildings = buildings[buildings.geometry.geom_type.isin(["Polygon", "MultiPolygon"])]
buildings = ox.projection.project_gdf(buildings, to_crs=crs)


def numeric_tag(gdf, tag):
    """Parse the leading number of an OSM tag (e.g. "45 m" -> 45.0), NaN if absent."""
    if tag not in gdf.columns:
        return pd.Series(np.nan, index=gdf.index)
    return pd.to_numeric(gdf[tag].astype(str).str.extract(r"(\d+\.?\d*)")[0], errors="coerce")


meters_per_level = 3.5
default_height = 15.0

height = numeric_tag(buildings, "height")
levels = numeric_tag(buildings, "building:levels")
buildings["height_m"] = height.fillna(levels * meters_per_level).fillna(default_height)
buildings["height_source"] = np.where(
    height.notna(), "height tag", np.where(levels.notna(), "building:levels", "default")
)
buildings["height_source"].value_counts()

In [ ]:
# plot building footprints colored by height, with the street network on top
fig, ax = plt.subplots(figsize=(8, 8))
buildings.plot(ax=ax, column="height_m", cmap="viridis", legend=True)
fig, ax = ox.plot.plot_graph(
    Gp, ax=ax, node_size=0, edge_color="#333333", edge_linewidth=0.8, bgcolor="w", show=False
)
ax.set_title("Building height (m)")
plt.show()

## 3. A route along the streets

Snap an origin and a destination to their nearest network nodes, then solve the shortest path. `route_to_gdf` returns the route's edges in order, so joining their geometries gives the full route as one line in meters.

In [ ]:
# origin near Portsmouth Square, destination near the Embarcadero (lng, lat)
orig = ox.distance.nearest_nodes(G, X=-122.4058, Y=37.7948)
dest = ox.distance.nearest_nodes(G, X=-122.3990, Y=37.7960)
route = ox.routing.shortest_path(Gp, orig, dest, weight="length")
route_edges = ox.routing.route_to_gdf(Gp, route)

# join the edge geometries (each runs from u to v) into one continuous line
coords = []
for geom in route_edges.geometry:
    pts = list(geom.coords)
    coords.extend(pts[1:] if coords and coords[-1] == pts[0] else pts)
route_line = LineString(coords)

straight = Point(coords[0]).distance(Point(coords[-1]))
print(f"Route along streets: {route_line.length:,.0f} m")
print(f"Straight line: {straight:,.0f} m (the route is {route_line.length / straight:.2f}x longer)")

## 4. Waypoints and a safe flight altitude

Place a waypoint every `spacing` meters along the route, then work out how high the drone must fly at each one:

1. **Ground elevation**: sample the elevation rasters at each waypoint.
2. **Obstacles**: find the tallest building whose footprint lies within `lateral_buffer` meters of the waypoint.
3. **Required height above ground**: the larger of the normal cruise height and that building's height plus a clearance margin.
4. **Climb limit**: a drone can't jump straight up at a building's edge, so we raise the profile wherever it would climb or descend more steeply than `max_climb_grade`. The profile is only ever raised, never lowered, so every clearance still holds.

In [ ]:
spacing = 10  # approximate meters between waypoints
cruise_agl = 30  # normal height above ground (m)
clearance = 10  # minimum margin above nearby buildings (m)
lateral_buffer = 15  # how far to the side we look for buildings (m)
max_climb_grade = 0.5  # max climb or descent: meters of altitude per meter flown

# waypoints along the route, in the projected CRS and in lat/lon
wp = gpd.GeoSeries(
    [Point(xy) for xy in ox.utils_geo.interpolate_points(route_line, spacing)], crs=crs
)
wp_latlon = wp.to_crs(4326)
along = np.array([route_line.project(p) for p in wp])

# 1. ground elevation from the same rasters, mosaicked in memory
mosaic, transform = merge(raster_paths)
rows, cols = rasterio.transform.rowcol(transform, wp_latlon.x, wp_latlon.y)
ground = mosaic[0][np.array(rows), np.array(cols)].astype(float)

# 2. tallest building near each waypoint (0 if there are none)
wp_idx, bldg_idx = buildings.sindex.query(wp.buffer(lateral_buffer), predicate="intersects")
heights = buildings["height_m"].to_numpy()
obstacle = np.zeros(len(wp))
np.maximum.at(obstacle, wp_idx, heights[bldg_idx])

# 3. required height above ground at each waypoint
required_agl = np.maximum(cruise_agl, obstacle + clearance)


# 4. enforce the climb/descent limit on the absolute (above sea level) altitude
def limit_grade(alt, dist_along, grade):
    alt = alt.copy()
    max_change = np.diff(dist_along) * grade  # allowed change between consecutive waypoints
    for i in range(1, len(alt)):  # don't descend too fast
        alt[i] = max(alt[i], alt[i - 1] - max_change[i - 1])
    for i in range(len(alt) - 2, -1, -1):  # start climbing early enough
        alt[i] = max(alt[i], alt[i + 1] - max_change[i])
    return alt


flight_amsl = limit_grade(ground + required_agl, along, max_climb_grade)
flight_agl = flight_amsl - ground
assert (flight_agl >= required_agl - 1e-9).all()

waypoints = gpd.GeoDataFrame(
    {
        "dist_m": along,
        "ground_m": ground,
        "obstacle_m": obstacle,
        "agl_m": flight_agl,
        "amsl_m": flight_amsl,
        "lat": wp_latlon.y,
        "lon": wp_latlon.x,
    },
    geometry=wp,
)
waypoints.describe().round(1)

In [ ]:
# altitude profile along the route
fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(along, ground.min() - 5, ground, color="#8c6d46", label="ground")
ax.fill_between(
    along, ground, ground + obstacle, step="mid", color="#999999", label="nearby buildings"
)
ax.plot(along, flight_amsl, color="#d62728", lw=2, label="flight path")
ax.set_xlabel("distance along route (m)")
ax.set_ylabel("elevation (m above sea level)")
ax.set_ylim(bottom=ground.min() - 5)
ax.legend(loc="upper left")
plt.show()

In [ ]:
# plot the route's waypoints on the map, colored by height above ground
fig, ax = plt.subplots(figsize=(8, 8))
buildings.plot(ax=ax, color="#cccccc")
fig, ax = ox.plot.plot_graph(
    Gp, ax=ax, node_size=0, edge_color="#999999", edge_linewidth=0.6, bgcolor="w", show=False
)
waypoints.plot(ax=ax, column="agl_m", cmap="plasma", markersize=12, legend=True, zorder=3)
ax.set_title("Flight height above ground (m)")
plt.show()

## 5. Prefer routes past low buildings

Section 4 adapts the altitude to whatever route we picked. We can also turn this around and pick the route with the obstacles in mind. Give every edge an `obstacle_m` attribute (the tallest building beside it), define an impedance that penalizes tall corridors, and route on that. This is the same pattern the [elevations example](12-node-elevations-edge-grades.ipynb) uses to avoid steep streets.

In [ ]:
edges = ox.convert.graph_to_gdfs(Gp, nodes=False)
edge_pos, bldg_pos = buildings.sindex.query(
    edges.geometry.buffer(lateral_buffer), predicate="intersects"
)
edge_obstacle = np.zeros(len(edges))
np.maximum.at(edge_obstacle, edge_pos, heights[bldg_pos])

# penalize length by how far the corridor's buildings rise above cruise height
penalty = 1 + np.maximum(edge_obstacle - cruise_agl, 0) / 10
impedances = edges["length"].to_numpy() * penalty
for (u, v, k), obstacle_m, impedance in zip(edges.index, edge_obstacle, impedances, strict=True):
    Gp.edges[u, v, k]["obstacle_m"] = obstacle_m
    Gp.edges[u, v, k]["low_impedance"] = impedance

route_low = ox.routing.shortest_path(Gp, orig, dest, weight="low_impedance")


def route_summary(route):
    gdf = ox.routing.route_to_gdf(Gp, route)
    return gdf["length"].sum(), gdf["obstacle_m"].max()


for name, r in [("shortest", route), ("low-building", route_low)]:
    length, tallest = route_summary(r)
    print(f"{name:>13}: {length:,.0f} m long, tallest building beside it {tallest:.0f} m")

In [ ]:
fig, ax = ox.plot.plot_graph_routes(
    Gp, [route, route_low], route_colors=["r", "y"], route_linewidth=4, node_size=0
)

## 6. Export the waypoints

Save the waypoints as 3D points (longitude, latitude, altitude above sea level) in GeoJSON. Most mission-planning tools can import them or be scripted to. Note the vertical reference: altitudes here come from the elevation raster, so they are relative to that raster's datum.

In [ ]:
out = gpd.GeoDataFrame(
    waypoints.drop(columns="geometry"),
    geometry=gpd.points_from_xy(waypoints["lon"], waypoints["lat"], waypoints["amsl_m"]),
    crs="epsg:4326",
)
out.to_file("./data/drone_waypoints.geojson", driver="GeoJSON")
out[["lat", "lon", "agl_m", "amsl_m"]].head()

### Where to go next

  - Fill in missing building heights with a better source (local open data, LiDAR) instead of the default.
  - Use `network_type="all"` or a custom filter if footpaths and parks are also acceptable corridors.
  - Add other features as obstacles or no-fly areas, e.g. `features_from_address(address, {"aeroway": True, "leisure": "park"}, dist)`.
  - Combine the obstacle penalty with grade, as in the [elevations example](12-node-elevations-edge-grades.ipynb), for ground robots that also need to follow streets.